# Jira Meeting Preparation Notebook

**Workflow:**
1. Configure search parameters
2. Search Jira & preview issues
3. Analyse (stats, risks, workload)
4. Generate AI executive summary & action items
5. Render & save meeting document

> **Prerequisites:** `.env` must contain `JIRA_URL`, `JIRA_EMAIL`, `JIRA_API_TOKEN`, and `ANTHROPIC_API_KEY`.

In [ ]:
import sys, os
from pathlib import Path

# Add src/ to path
repo_root = Path(os.getcwd()).parent
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from jira_briefing import JiraClient, analyze, JiraSummarizer, MeetingDocGenerator, SearchConfig, MeetingConfig
print('Imports OK')

## Step 1 — Configure Search
Edit the `SearchConfig` below to target the issues you care about.

In [ ]:
search_config = SearchConfig(
    # --- Option A: structured filters ---
    projects=["QUAL"],           # Jira project keys
    sprint_state="active",       # "active" | "closed" | "future"
    # sprint="Sprint 24",        # exact sprint name (overrides sprint_state)
    # assignees=["Kunwoo Lee"],   # filter by assignee
    # statuses=["In Progress", "Blocked"],
    # priorities=["High", "Critical"],
    # updated_days=14,            # issues updated in last 14 days
    # keywords="warranty defect",

    # --- Option B: raw JQL override (ignores structured filters above) ---
    # jql="project = QUAL AND sprint in openSprints() ORDER BY priority",

    max_results=200,
    include_comments=False,
)

meeting_config = MeetingConfig(
    title="Sprint Status Review",
    author="Kunwoo Lee",
    output_format="markdown",    # "markdown" | "html"
)

print('Config ready.')

## Step 2 — Search Jira

In [ ]:
client = JiraClient()
jql = client.build_jql(search_config)
print(f'JQL: {jql}\n')

issues = client.search(search_config)
print(f'Retrieved {len(issues)} issues.')

In [ ]:
# Preview first 10 issues
import pandas as pd

preview = pd.DataFrame([
    {
        'Key': i.key,
        'Type': i.issue_type,
        'Priority': i.priority,
        'Status': i.status,
        'Assignee': i.assignee or '—',
        'Sprint': i.sprint or '—',
        'SP': i.story_points,
        'Due': i.due_date.strftime('%Y-%m-%d') if i.due_date else '—',
        'Summary': i.summary[:60],
    }
    for i in issues[:10]
])
display(preview)

## Step 3 — Analyse

In [ ]:
result = analyze(issues, config=search_config)

print(f'Total issues : {result.total_issues}')
print(f'Risks flagged: {len(result.risks)}')
print(f'Overdue      : {len(result.overdue_issues)}')
print()
print('Status distribution:')
for s, c in sorted(result.by_status.items(), key=lambda x: -x[1]):
    print(f'  {s:<25} {c}')

In [ ]:
# Sprint dashboard (if available)
if result.sprint_summary:
    s = result.sprint_summary
    print(f'Sprint : {s.name}')
    print(f'Done   : {s.done_issues}/{s.total_issues} issues ({s.completion_pct}%)')
    if s.total_story_points:
        print(f'SP     : {s.done_story_points:.0f}/{s.total_story_points:.0f} ({s.sp_completion_pct}%)')
    print(f'Blocked: {s.blocked_issues}')
else:
    print('No sprint data found in retrieved issues.')

In [ ]:
# Top risks
risk_df = pd.DataFrame([
    {'Severity': r.severity, 'Type': r.risk_type, 'Key': r.issue_key,
     'Assignee': r.assignee or '—', 'Detail': r.detail[:80]}
    for r in result.risks[:15]
])
if not risk_df.empty:
    display(risk_df)
else:
    print('No risks detected.')

In [ ]:
# Workload distribution chart
import matplotlib.pyplot as plt

if result.by_assignee:
    names = [w.assignee[:20] for w in result.by_assignee[:10]]
    totals = [w.total for w in result.by_assignee[:10]]
    overdues = [w.overdue for w in result.by_assignee[:10]]

    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.barh(names[::-1], totals[::-1], color='steelblue', label='Total')
    ax.barh(names[::-1], overdues[::-1], color='tomato', label='Overdue')
    ax.set_xlabel('Issue Count')
    ax.set_title('Team Workload (Top 10 Assignees)')
    ax.legend()
    plt.tight_layout()
    plt.show()

## Step 4 — AI Summary (Claude)
This cell calls the Claude API. Set `skip_ai = True` to bypass.

In [ ]:
skip_ai = False   # set True to skip API call

ai_output = {"executive_summary": "", "action_items": ""}

if not skip_ai:
    summarizer = JiraSummarizer()
    ai_output = summarizer.summarize(result, issues, meeting_config)
    print('=== Executive Summary ===')
    print(ai_output['executive_summary'])
    print()
    print('=== Action Items ===')
    print(ai_output['action_items'])
else:
    print('AI summary skipped.')

## Step 5 — Render & Save Meeting Document

In [ ]:
generator = MeetingDocGenerator(result, issues, ai_output, meeting_config)
out_path = generator.save()
print(f'Saved to: {out_path}')

In [ ]:
# Preview the first 100 lines of the document
doc = out_path.read_text(encoding='utf-8')
for line in doc.splitlines()[:100]:
    print(line)